In [1]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
con = duckdb.connect('../data/mt2.duckdb')
runs = con.execute("""
    SELECT
        *
    FROM stg_runs
""").df()

cards = con.execute("""
    SELECT
        *
    FROM stg_cards
""").df()

cards_per_run = con.execute("""
    SELECT
        *
    FROM fct_run_card_stats
""").df()


con.close()

In [5]:
runs.head()

,run_id,run_date,primary_clan,primary_level,secondary_clan,secondary_level,hero,covenant_rank,pyre_heart,score,run_time,result,battle_1,battle_2,battle_3,battle_4,battle_5,battle_6,battle_7,battle_8
0,1,2025-12-20,Banished,1,Pyreborne,1,Fel,0,Proto Heartcage,21245,00:54:41,Loss,Dominion's Preservation,Knights of Entropy,"Arkion, Scion of Savagery",Penance Seekers,Relentless Zealots,"Cael, Lord of the Cherubs",Savagery's Final Assault,NaN
1,2,2025-12-20,Banished,2,Pyreborne,2,Fel,1,Proto Heartcage,15285,00:55:30,Loss,Dominion's Preservation,Quicksilver Cherubs,Corrupted Mass,Hordes of Heaven,Relentless Zealots,Stryx & Hallow,NaN,NaN
2,3,2025-12-22,Banished,3,Pyreborne,2,Fel,1,Proto Heartcage,15682,01:11:01,Loss,Dominion's Preservation,March of the Supplicants,The Fleshweaver,Savagery's Hunters,Corrupted Crusade,Stryx & Hallow,Savagery's Final Assault,NaN
3,4,2025-12-23,Banished,3,Pyreborne,2,Fel,0,Proto Heartcage,16976,00:43:07,Loss,Heavenly Toll,Flagellant's March,"Arkion, Scion of Savagery",Savagery's Hunters,Undying Minions,Stryx & Hallow,NaN,NaN
4,5,2025-12-28,Banished,4,Pyreborne,3,Fel,0,Wyngh's Spirit,36635,01:18:59,Victory,Heavenly Toll,Quicksilver Cherubs,"Arkion, Scion of Savagery",Dire Defenses,Undying Minions,"Cael, Lord of the Cherubs",Savagery's Final Assault,"Seraph Aeternus, Chosen of Dominion"


In [3]:
#lets subset to Banished runs where i won or i got to battle_8

banished_ring8=runs[(runs['battle_8'].notna()) & ((runs['primary_clan']=='Banished') | (runs['secondary_clan']=='Banished'))]

In [4]:
banished_ring8.info()

<class 'pandas.DataFrame'>
Index: 12 entries, 4 to 59
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   run_id           12 non-null     int64         
 1   run_date         12 non-null     datetime64[us]
 2   primary_clan     12 non-null     str           
 3   primary_level    12 non-null     int32         
 4   secondary_clan   12 non-null     str           
 5   secondary_level  12 non-null     int32         
 6   hero             12 non-null     str           
 7   covenant_rank    12 non-null     int32         
 8   pyre_heart       12 non-null     str           
 9   score            12 non-null     int32         
 10  run_time         12 non-null     str           
 11  result           12 non-null     str           
 12  battle_1         12 non-null     str           
 13  battle_2         12 non-null     str           
 14  battle_3         12 non-null     str           
 15  battle_

In [5]:

banished_1=banished_ring8[['run_id','primary_clan','secondary_clan','hero','covenant_rank','pyre_heart','run_time','result']]

In [6]:
banished_1

,run_id,primary_clan,secondary_clan,hero,covenant_rank,pyre_heart,run_time,result
4,5,Banished,Pyreborne,Fel,0,Wyngh's Spirit,01:18:59,Victory
9,10,Banished,Pyreborne,Fel,1,Wyngh's Spirit,01:21:44,Victory
10,11,Banished,Pyreborne,Fel,1,Wyngh's Spirit,01:21:07,Victory
16,17,Underlegion,Banished,Bolete,1,Wyngh's Spirit,01:10:00,Victory
23,24,Luna Coven,Banished,Arduhn,1,Wyngh's Spirit,01:22:17,Loss
29,30,Lazarus League,Banished,Orechi,1,Bogwurm's Growth,01:26:16,Victory
34,35,Banished,Awoken,Fel,1,Bogwurm's Growth,01:27:23,Victory
44,45,Banished,Stygian Guard,Talos,1,Fhyra's Greed,01:15:40,Victory
45,46,Banished,Underlegion,Fel,1,Fhyra's Greed,00:54:58,Victory
47,48,Banished,Hellhorned,Talos,2,Wyngh's Spirit,00:52:10,Victory


In [7]:
banished_cards=banished_1.merge(cards_per_run,on='run_id',how='left')

In [8]:
banished_cards

,run_id,primary_clan,secondary_clan,hero,covenant_rank,pyre_heart,run_time,result,num_cards,num_upgrades,...,stygian_guard_cards,umbra_cards,melting_remnant_cards,wurmkin_cards,banished_cards,pyreborne_cards,luna_coven_cards,underlegion_cards,lazarus_league_cards,railforged_cards
0,5,Banished,Pyreborne,Fel,0,Wyngh's Spirit,01:18:59,Victory,20.0,13.0,...,0.0,0.0,0.0,0.0,10.0,7.0,0.0,0.0,0.0,0.0
1,10,Banished,Pyreborne,Fel,1,Wyngh's Spirit,01:21:44,Victory,20.0,18.0,...,0.0,0.0,0.0,0.0,13.0,5.0,0.0,0.0,0.0,0.0
2,11,Banished,Pyreborne,Fel,1,Wyngh's Spirit,01:21:07,Victory,24.0,10.0,...,0.0,0.0,0.0,0.0,10.0,8.0,0.0,0.0,0.0,0.0
3,17,Underlegion,Banished,Bolete,1,Wyngh's Spirit,01:10:00,Victory,25.0,13.0,...,0.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0
4,24,Luna Coven,Banished,Arduhn,1,Wyngh's Spirit,01:22:17,Loss,30.0,11.0,...,0.0,0.0,0.0,0.0,11.0,0.0,7.0,0.0,0.0,0.0
5,30,Lazarus League,Banished,Orechi,1,Bogwurm's Growth,01:26:16,Victory,28.0,9.0,...,0.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0
6,35,Banished,Awoken,Fel,1,Bogwurm's Growth,01:27:23,Victory,25.0,11.0,...,0.0,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0
7,45,Banished,Stygian Guard,Talos,1,Fhyra's Greed,01:15:40,Victory,33.0,18.0,...,0.0,0.0,0.0,0.0,11.0,0.0,0.0,0.0,0.0,0.0
8,46,Banished,Underlegion,Fel,1,Fhyra's Greed,00:54:58,Victory,26.0,20.0,...,0.0,0.0,0.0,0.0,14.0,0.0,0.0,0.0,0.0,0.0
9,48,Banished,Hellhorned,Talos,2,Wyngh's Spirit,00:52:10,Victory,30.0,22.0,...,0.0,0.0,0.0,0.0,13.0,0.0,0.0,0.0,0.0,0.0


In [18]:
banished_cards=banished_1.merge(cards,on='run_id',how='left')

In [19]:
banished_cards

,run_id,primary_clan,secondary_clan,hero,covenant_rank,pyre_heart,run_time,result,card_type,card_name,copies,upgrades
0,5,Banished,Pyreborne,Fel,0,Wyngh's Spirit,01:18:59,Victory,Equipment,Everlasting Light,1,0
1,5,Banished,Pyreborne,Fel,0,Wyngh's Spirit,01:18:59,Victory,Equipment,Mind Cage,1,0
2,5,Banished,Pyreborne,Fel,0,Wyngh's Spirit,01:18:59,Victory,Equipment,Stoic Platemail,1,0
3,5,Banished,Pyreborne,Fel,0,Wyngh's Spirit,01:18:59,Victory,Equipment,Windblade,1,0
4,5,Banished,Pyreborne,Fel,0,Wyngh's Spirit,01:18:59,Victory,Room,Bloodsoaked Arena,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
248,60,Awoken,Banished,Wyldenten,3,Bogwurm's Growth,01:24:13,Loss,Spell,Heavensward,1,0
249,60,Awoken,Banished,Wyldenten,3,Bogwurm's Growth,01:24:13,Loss,Spell,Machinists Tome,1,0
250,60,Awoken,Banished,Wyldenten,3,Bogwurm's Growth,01:24:13,Loss,Spell,Pyre Go,1,0
251,60,Awoken,Banished,Wyldenten,3,Bogwurm's Growth,01:24:13,Loss,Spell,Rootseeds,4,0
